In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from langchain_core.documents import Document

# Load medicine datasets
df1 = pd.read_csv("data/A_Z_medicines_dataset_of_india.csv")
df2 = pd.read_csv("data/medDataset_processed.csv")

df1_clean = df1[['name','price(₹)','manufacturer_name','type',
                  'pack_size_label','short_composition1',
                  'short_composition2']].copy()
df1_clean.fillna("Not available", inplace=True)
df1_clean['text'] = df1_clean.apply(lambda row:
    f"Medicine: {row['name']}\n"
    f"Price: ₹{row['price(₹)']}\n"
    f"Manufacturer: {row['manufacturer_name']}\n"
    f"Type: {row['type']}\n"
    f"Pack Size: {row['pack_size_label']}\n"
    f"Composition: {row['short_composition1']}, {row['short_composition2']}",
    axis=1)

df2_clean = df2[['Question','Answer']].dropna()
df2_clean['text'] = df2_clean.apply(lambda row:
    f"Question: {row['Question']}\nAnswer: {row['Answer']}", axis=1)

# Load medical stores dataset
stores_df = pd.read_excel("data/jaipur_medical_stores_ml.xlsx")
stores_df['Latitude'] = pd.to_numeric(stores_df['Latitude'], errors='coerce')
stores_df['Longitude'] = pd.to_numeric(stores_df['Longitude'], errors='coerce')
stores_df = stores_df.dropna(subset=['Latitude', 'Longitude'])

print(f"✅ Medicines loaded: {len(df1_clean)}")
print(f"✅ Q&A loaded: {len(df2_clean)}")
print(f"✅ Medical stores loaded: {len(stores_df)}")
print(f"📍 Localities covered: {stores_df['Locality'].nunique()}")

✅ Medicines loaded: 253973
✅ Q&A loaded: 16407
✅ Medical stores loaded: 383
📍 Localities covered: 40


In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory="vectorstore",
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

print(f"✅ Vectorstore loaded!")
print(f"📊 Total documents: {vectorstore._collection.count()}")

# Quick test
results = vectorstore.similarity_search("paracetamol", k=2)
print(f"\n🧪 Search test:")
for r in results:
    print(f"→ {r.page_content[:80]}")

✅ Vectorstore loaded!
📊 Total documents: 684568

🧪 Search test:
→ Medicine: Paragreat 250mg Suspension
Price: ₹44.35
Manufacturer: Mankind Pharma 
→ Medicine: Paragreat 650mg Tablet
Price: ₹30.05
Manufacturer: Mankind Pharma Ltd



In [4]:
from langchain_community.llms import Ollama
from geopy.distance import geodesic

# Load LLM
llm = Ollama(model="mistral", temperature=0.3)
print("✅ Mistral loaded!")

# Find nearest stores function
def find_nearest_stores(user_area=None, user_lat=None, 
                         user_lon=None, top_n=3):
    df = stores_df.copy()
    
    if user_lat and user_lon:
        # GPS based
        def calc_dist(row):
            return geodesic(
                (user_lat, user_lon),
                (row['Latitude'], row['Longitude'])
            ).km
        df['distance_km'] = df.apply(calc_dist, axis=1)
        df = df.sort_values('distance_km')
    
    elif user_area:
        # Area name based
        area_lower = user_area.lower()
        df['score'] = (
            df['Locality'].str.lower().str.contains(area_lower, na=False).astype(int) +
            df['Address'].str.lower().str.contains(area_lower, na=False).astype(int) +
            df['Zone'].str.lower().str.contains(area_lower, na=False).astype(int)
        )
        df = df[df['score'] > 0].sort_values('score', ascending=False)
        df['distance_km'] = 0
    
    results = []
    for _, row in df.head(top_n).iterrows():
        # Phone cleanup
        phone = str(row.get('Phone Primary', 'N/A'))
        if phone.startswith('91') and len(phone) >= 12:
            phone = phone[2:]
        
        # Delivery
        has_delivery = row.get('Has Delivery', False)
        delivery_radius = row.get('Delivery Radius Km', None)
        if has_delivery == True:
            delivery = f"🚚 Delivery: {delivery_radius}km radius" if pd.notna(delivery_radius) else "🚚 Delivery available"
        else:
            delivery = "❌ No delivery"
        
        # Timing
        is_24x7 = row.get('Is 24X7', False)
        if is_24x7 == True:
            timing = "🕐 Open 24x7"
        else:
            timing = f"🕐 {row.get('Opening Time','N/A')} - {row.get('Closing Time','N/A')}"
        
        # Distance
        dist = row.get('distance_km', 0)
        dist_text = f"{dist:.1f}km away" if dist > 0 else row.get('Locality','N/A')
        
        # Rating
        rating = row.get('Google Rating', 'N/A')
        reviews = int(row.get('Review Count', 0))
        
        store_text = (
            f"🏥 {row['Store Name']} ({dist_text})\n"
            f"   ⭐ {rating}/5 ({reviews} reviews)\n"
            f"   {timing}\n"
            f"   {delivery}\n"
            f"   📞 {phone}\n"
            f"   📬 {row.get('Address','N/A')}\n"
            f"   🗺️ {row.get('Google Maps Link','N/A')}"
        )
        results.append(store_text)
    
    return results

# Q&A with location
def ask_with_location(question, area=None):
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    
    store_section = ""
    if area and area.strip():
        stores = find_nearest_stores(user_area=area, top_n=3)
        if stores:
            store_section = "\n\nNEARBY MEDICAL STORES IN " + area.upper() + ":\n"
            store_section += "\n\n".join(stores)
        else:
            store_section = f"\n\nNo stores found in {area}. Try nearby areas like Malviya Nagar, Vaishali Nagar, C-Scheme."
    
    prompt = f"""You are MediBot Jaipur, a helpful medical assistant for Jaipur city.
Use the medicine context below to answer the question clearly.
List specific medicine names, prices and compositions.
Then show the nearby medical stores.
Always remind users to consult a real doctor.

MEDICINE INFORMATION:
{context}
{store_section}

Question: {question}
User Area: {area if area else 'Not specified'}

Answer:"""
    
    return llm.invoke(prompt)

print("✅ Location functions ready!")

# Quick test
print("\n🧪 Testing store search...")
test_stores = find_nearest_stores(user_area="Malviya Nagar", top_n=2)
for s in test_stores:
    print(s)
    print()

✅ Mistral loaded!
✅ Location functions ready!

🧪 Testing store search...
🏥 Apollo Pharmacy (Malviya Nagar)
   ⭐ 3.4/5 (302 reviews)
   🕐 07:00 - 23:00
   ❌ No delivery
   📞 9848870700
   📬 Shop No. 243, Malviya Nagar Colony, Malviya Nagar, Jaipur - 302004
   🗺️ https://maps.google.com/?q=26.862531,75.811106

🏥 City Med House (Malviya Nagar)
   ⭐ 4.7/5 (2790 reviews)
   🕐 07:00 - 21:00
   🚚 Delivery: 5.0km radius
   📞 8163907779
   📬 Shop No. 44, Malviya Nagar Circle, Malviya Nagar, Jaipur - 302018
   🗺️ https://maps.google.com/?q=26.861947,75.803862



C:\Users\alpha\AppData\Local\Temp\ipykernel_11768\622542179.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral", temperature=0.3)


In [5]:
import gradio as gr
from gradio import ChatMessage

def ask_assistant(question, area, history):
    if not question.strip():
        return history, ""
    response = ask_with_location(question, area=area)
    display_q = f"📍 {area} | {question}" if area.strip() else question
    history = history + [
        ChatMessage(role="user", content=display_q),
        ChatMessage(role="assistant", content=response)
    ]
    return history, ""

def clear_chat():
    return [], ""

css = """
body { background: linear-gradient(135deg, #0f0c29, #302b63, #24243e) !important; }
footer { display: none !important }
"""

with gr.Blocks(css=css, title="💊 MediBot Jaipur") as demo:
    gr.HTML("""
    <div style="text-align:center; padding:30px 0 10px 0;">
        <div style="font-size:3em;">💊</div>
        <div style="font-size:2.2em; font-weight:900;
                    background:linear-gradient(90deg, #00c6ff, #0072ff);
                    -webkit-background-clip:text;
                    -webkit-text-fill-color:transparent;">
            MediBot Jaipur
        </div>
        <div style="color:#a0aec0; margin-top:5px;">
            Medicines + Nearest Medical Stores in Jaipur 🏙️
        </div>
        <div style="margin-top:10px; padding:8px 20px;
                    background:rgba(255,100,100,0.15);
                    border:1px solid #ff6464; border-radius:20px;
                    display:inline-block; color:#ff6464; font-size:0.9em;">
            ⚠️ Educational purposes only — Always consult a real doctor!
        </div>
    </div>
    """)
    gr.HTML("""
    <div style="display:flex; justify-content:center; gap:15px; margin:20px 0;">
        <div style="background:rgba(0,198,255,0.1); border:1px solid #00c6ff;
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#00c6ff;">383</div>
            <div style="color:#a0aec0; font-size:0.85em;">Jaipur Stores</div>
        </div>
        <div style="background:rgba(0,198,255,0.1); border:1px solid #00c6ff;
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#00c6ff;">253K+</div>
            <div style="color:#a0aec0; font-size:0.85em;">Indian Medicines</div>
        </div>
        <div style="background:rgba(0,198,255,0.1); border:1px solid #00c6ff;
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#00c6ff;">16K+</div>
            <div style="color:#a0aec0; font-size:0.85em;">Medical Q&As</div>
        </div>
        <div style="background:rgba(118,75,162,0.2); border:1px solid #764ba2;
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#b794f4;">100%</div>
            <div style="color:#a0aec0; font-size:0.85em;">Offline & Free</div>
        </div>
    </div>
    """)
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=350,
                label="💬 Chat with MediBot Jaipur"
            )
            area_box = gr.Textbox(
                placeholder="📍 Your area in Jaipur (e.g. Malviya Nagar, Vaishali Nagar, C-Scheme...)",
                label="📍 Your Location"
            )
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="💊 Ask about any medicine...",
                    label="", scale=5, container=False
                )
                submit_btn = gr.Button("Ask 🚀", variant="primary", scale=1)
            clear_btn = gr.Button("🗑️ Clear Chat", size="sm")

        with gr.Column(scale=1):
            gr.Markdown("### ⚡ Quick Questions")
            q1 = gr.Button("💊 Paracetamol uses?", size="sm")
            q2 = gr.Button("💰 Price of Augmentin 625?", size="sm")
            q3 = gr.Button("🤒 Medicines for fever?", size="sm")
            q4 = gr.Button("⚠️ Azithromycin side effects?", size="sm")
            q5 = gr.Button("🔬 Medicines with Amoxicillin?", size="sm")
            q6 = gr.Button("😴 Medicines for sleep?", size="sm")
            gr.Markdown("---")
            gr.Markdown("""
### 📍 Popular Areas
Malviya Nagar
Vaishali Nagar
C-Scheme
Mansarovar
Tonk Road
Sanganer
Jagatpura
Ajmer Road
            """)
            gr.Markdown("""
### 🔧 Tech Stack
🤖 **LLM:** Mistral 7B
🔍 **Search:** MiniLM
🗄️ **DB:** ChromaDB
📍 **Location:** GeoPy
🦜 **Framework:** LangChain
            """)

    submit_btn.click(
        ask_assistant,
        inputs=[question_box, area_box, chatbot],
        outputs=[chatbot, question_box]
    )
    question_box.submit(
        ask_assistant,
        inputs=[question_box, area_box, chatbot],
        outputs=[chatbot, question_box]
    )
    clear_btn.click(clear_chat, outputs=[chatbot, question_box])
    q1.click(lambda a, h: ask_assistant("What is Paracetamol used for?", a, h), inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q2.click(lambda a, h: ask_assistant("Price of Augmentin 625?", a, h), inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q3.click(lambda a, h: ask_assistant("What medicines are used for fever?", a, h), inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q4.click(lambda a, h: ask_assistant("Side effects of Azithromycin?", a, h), inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q5.click(lambda a, h: ask_assistant("What medicines contain Amoxicillin?", a, h), inputs=[area_box, chatbot], outputs=[chatbot, question_box])
    q6.click(lambda a, h: ask_assistant("What medicines help with sleep?", a, h), inputs=[area_box, chatbot], outputs=[chatbot, question_box])

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
